# Multiple Linear Regression Practical - Economics Dataset

**Goal:** predict `index_price` using two inputs: `interest_rate` and `unemployment_rate`. This is **multiple** linear regression because there is more than one input feature.

## 1. What we will do

1. Load the supplied economics data.
2. Check missing values and simple relationships.
3. Train a multiple linear regression model.
4. Test it on unseen rows and read the metrics.
5. See the prediction plane and residuals.
6. Verify the result with a manual OLS calculation.

The supplied file contains a saved row-number column plus year and month. This lesson deliberately uses only interest rate and unemployment rate, so those extra columns are left out.

## 2. Load the data

The 24 usable rows from the supplied CSV are included below so this notebook runs after a GitHub clone without a local Downloads path.

- `interest_rate` and `unemployment_rate` are inputs ($X$).
- `index_price` is the numeric output we want to predict ($y$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

# Values copied from economic_index.csv after removing row number, year, and month
data = {
    'interest_rate': [2.75, 2.5, 2.5, 2.5, 2.5, 2.5, 2.5, 2.25, 2.25, 2.25, 2.0, 2.0, 2.0, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75],
    'unemployment_rate': [5.3, 5.3, 5.3, 5.3, 5.4, 5.6, 5.5, 5.5, 5.5, 5.6, 5.7, 5.9, 6.0, 5.9, 5.8, 6.1, 6.2, 6.1, 6.1, 6.1, 5.9, 6.2, 6.2, 6.1],
    'index_price': [1464, 1394, 1357, 1293, 1256, 1254, 1234, 1195, 1159, 1167, 1130, 1075, 1047, 965, 943, 958, 971, 949, 884, 866, 876, 822, 704, 719]
}
df = pd.DataFrame(data)

print('Rows, columns:', df.shape)
print('Missing values:')
print(df.isna().sum())
df.head()


## 3. Look at the relationships

The two plots show each input against the target. Correlation tells us whether two numbers tend to move together, but it does **not** prove cause and effect.

Notice one important warning: the two input features are strongly related to each other in this tiny dataset. This is called **multicollinearity**. The model can still predict, but the individual coefficient values can change a lot if the data changes a little.

In [ ]:
correlation = df.corr().round(3)
print('Correlation table:')
print(correlation)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(df['interest_rate'], df['index_price'], color='#1d3557', s=60)
axes[0].set_title('Interest rate vs index price', weight='bold')
axes[0].set_xlabel('Interest rate')
axes[0].set_ylabel('Index price')
axes[0].grid(alpha=0.2)

axes[1].scatter(df['unemployment_rate'], df['index_price'], color='#d62828', s=60)
axes[1].set_title('Unemployment rate vs index price', weight='bold')
axes[1].set_xlabel('Unemployment rate')
axes[1].set_ylabel('Index price')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## 4. Split and train
The model learns a plane, not a line:

`predicted index price = intercept + b_interest × interest rate + b_unemployment × unemployment rate`

- `X` is a 2D table with two feature columns.
- `y` is the 1D target column.
- 25% of rows are kept for testing.
- We do **not** scale the features here. Ordinary linear regression can learn from the original units, so the coefficients stay easier to read. If you use scaling in another model, fit it on training data only and use `transform` (not `fit_transform`) on test data.

In [ ]:
X = df[['interest_rate', 'unemployment_rate']]
y = df['index_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

intercept = float(model.intercept_)
interest_coefficient, unemployment_coefficient = model.coef_
print('Training feature shape:', X_train.shape)
print('Test feature shape:', X_test.shape)
print(f'Intercept: {intercept:.2f}')
print(f'Interest-rate coefficient: {interest_coefficient:.2f}')
print(f'Unemployment-rate coefficient: {unemployment_coefficient:.2f}')
print('Read carefully: each coefficient describes a change while the other feature is held fixed.')


## 5. Test the model and use cross-validation

Test metrics use only the six unseen test rows. Cross-validation repeats smaller train/validation checks inside the training data, so it gives more than one error estimate.

- **MAE:** average mistake in index-price units.
- **RMSE:** typical mistake size, with extra weight for large mistakes.
- **R-squared:** comparison with the average-price baseline.
- **Adjusted R-squared:** R-squared adjusted for the number of input features.

This dataset is very small, so every score is only a learning example. Do not use it to make economic decisions.

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
n_test_rows = len(y_test)
number_of_features = X_test.shape[1]
adjusted_r2 = 1 - (1 - r2) * (n_test_rows - 1) / (n_test_rows - number_of_features - 1)

# Cross-validation uses only the training portion; sklearn returns negative MSE by convention
cv_negative_mse = cross_val_score(
    LinearRegression(), X_train, y_train, scoring='neg_mean_squared_error', cv=3
)
cv_rmse = np.sqrt(-cv_negative_mse)

metrics = pd.Series({
    'Test MAE': mae,
    'Test MSE': mse,
    'Test RMSE': rmse,
    'Test R-squared': r2,
    'Test Adjusted R-squared': adjusted_r2,
    'Mean 3-fold CV RMSE (training data)': cv_rmse.mean()
}).round(3)
metrics


## 6. Visual diagram: actual values, predictions, and the regression plane

Left: points close to the dashed diagonal have accurate predictions. Right: the yellow plane combines the two inputs to predict index price; a plane is the multiple-regression version of a best-fit line.

In [ ]:
fig = plt.figure(figsize=(14, 5.2))

# Left: actual versus predicted values
ax1 = fig.add_subplot(121)
ax1.scatter(y_test, y_pred, color='#d62828', s=70, label='Test predictions')
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
ax1.plot([low, high], [low, high], '--', color='#1d3557', label='Perfect prediction')
ax1.set_title('Actual vs predicted index price', weight='bold')
ax1.set_xlabel('Actual index price')
ax1.set_ylabel('Predicted index price')
ax1.legend()
ax1.grid(alpha=0.2)

# Right: a multiple-regression plane uses two input axes
ax2 = fig.add_subplot(122, projection='3d')
interest_grid, unemployment_grid = np.meshgrid(
    np.linspace(df['interest_rate'].min(), df['interest_rate'].max(), 25),
    np.linspace(df['unemployment_rate'].min(), df['unemployment_rate'].max(), 25)
)
price_plane = intercept + interest_coefficient * interest_grid + unemployment_coefficient * unemployment_grid
ax2.plot_surface(interest_grid, unemployment_grid, price_plane, color='#ffd166', alpha=0.45, edgecolor='none')
ax2.scatter(X_train['interest_rate'], X_train['unemployment_rate'], y_train, color='#1d3557', s=45, label='Training data')
ax2.scatter(X_test['interest_rate'], X_test['unemployment_rate'], y_test, color='#d62828', s=55, label='Test data')
ax2.set_title('One plane for two input features', weight='bold', pad=15)
ax2.set_xlabel('Interest rate', labelpad=8)
ax2.set_ylabel('Unemployment rate', labelpad=8)
ax2.set_zlabel('Index price', labelpad=8)
ax2.legend(loc='upper left', fontsize=8)
ax2.view_init(elev=23, azim=-58)

plt.tight_layout()
plt.show()


## 7. OLS check without a special library

OLS (Ordinary Least Squares) solves the same squared-error problem. For multiple features, we add a first column of ones for the intercept, then use `np.linalg.lstsq` to find the best coefficients. This is safer than calculating a matrix inverse by hand.

The manual OLS values below should match scikit-learn, apart from tiny rounding differences.

In [ ]:
# Add a column of 1s so OLS can learn an intercept
X_train_with_intercept = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
ols_intercept, ols_interest, ols_unemployment = np.linalg.lstsq(
    X_train_with_intercept, y_train.to_numpy(), rcond=None
)[0]

print(f'Manual OLS intercept: {ols_intercept:.6f}')
print(f'scikit-learn intercept: {intercept:.6f}')
print(f'Manual OLS interest coefficient: {ols_interest:.6f}')
print(f'scikit-learn interest coefficient: {interest_coefficient:.6f}')
print(f'Manual OLS unemployment coefficient: {ols_unemployment:.6f}')
print(f'scikit-learn unemployment coefficient: {unemployment_coefficient:.6f}')
print('Do all values match?', np.allclose(
    [ols_intercept, ols_interest, ols_unemployment],
    [intercept, interest_coefficient, unemployment_coefficient]
))


## 8. Predict one new situation

To predict a new value, pass both inputs in a 2D table with the **same feature names and order** used during training. This example is only a code demonstration, not a real economic forecast.

In [ ]:
new_economy_data = pd.DataFrame({
    'interest_rate': [2.25],
    'unemployment_rate': [5.6]
})
new_prediction = model.predict(new_economy_data)[0]
print(f'Predicted index price: {new_prediction:.2f}')


## 9. Quick revision

- Multiple linear regression uses two or more input features.
- Keep features in a 2D table: `df[['interest_rate', 'unemployment_rate']]`.
- Test performance must use unseen data.
- A multiple-regression model fits a plane when there are two inputs.
- Very high correlation between input features is called multicollinearity; it makes individual coefficient interpretation less reliable.
- OLS and scikit-learn Linear Regression solve the same least-squares problem in this example.

**One-line answer:** Multiple linear regression combines the effects of several input features to predict one numeric output.